In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

import mlflow 
import mlflow.sklearn


In [3]:
RANDOM_STATE = 42
EXPERIMENT_NAME = "logistic_regression_training"

In [4]:
# MLFLOW Setup
mlflow.set_experiment(EXPERIMENT_NAME)

2025/12/30 21:39:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/30 21:39:30 INFO mlflow.store.db.utils: Updating database tables
2025/12/30 21:39:30 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/30 21:39:30 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/30 21:39:30 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2025/12/30 21:39:30 INFO alembic.runtime.migration: Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2025/12/30 21:39:30 INFO alembic.runtime.migration: Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2025/12/30 21:39:30 INFO alembic.runtime.migration: Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
2025/12/30 21:39:30 INFO alembic.runtime.migration: Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
2025/12/30 21:39:30 INFO alembic.runtime.migration: Running 

<Experiment: artifact_location=('file:c:/Users/Aman/OneDrive/Documents/GitHub/ML-Models/ML-EXPERIMENT '
 '1/notebooks/mlruns/1'), creation_time=1767110971810, experiment_id='1', last_update_time=1767110971810, lifecycle_stage='active', name='logistic_regression_training', tags={}>

In [14]:
dataPath = "../data/titanic.csv"
df = pd.read_csv(dataPath)
df.head()
df.shape


(891, 12)

In [19]:
TARGET_COL = "Survived"

NUMERIC_FEATURES = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

CATEGORICAL_FEATURES = [
    "Sex",
    "Embarked",
    "Pclass"
]

x = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET_COL]

x

,Age,SibSp,Parch,Fare,Sex,Embarked,Pclass
0,22.0,1,0,7.2500,male,S,3
1,38.0,1,0,71.2833,female,C,1
2,26.0,0,0,7.9250,female,S,3
3,35.0,1,0,53.1000,female,S,1
4,35.0,0,0,8.0500,male,S,3
...,...,...,...,...,...,...,...
886,27.0,0,0,13.0000,male,S,2
887,19.0,0,0,30.0000,female,S,1
888,NaN,1,2,23.4500,female,S,3
889,26.0,0,0,30.0000,male,C,1


In [20]:
xTrain, xTest, yTrain, yTest = train_test_split(
    x,
    y,
    random_state= RANDOM_STATE,
    test_size = 0.2,
    stratify=y
)

print(xTrain.shape, xTest.shape)

(712, 7) (179, 7)


In [36]:
# Save split for reuse
xTrain.to_csv("../data/X_train.csv", index=False)
xTest.to_csv("../data/X_test.csv", index=False)
yTrain.to_csv("../data/y_train.csv", index=False)
yTest.to_csv("../data/y_test.csv", index=False)

In [24]:
# Preprocessing 
print(df[NUMERIC_FEATURES].isna().sum())
print(df[CATEGORICAL_FEATURES].isna().sum())

Age      177
SibSp      0
Parch      0
Fare       0
dtype: int64
Sex         0
Embarked    2
Pclass      0
dtype: int64


### Why Median Imputation for Age?
- Age is right-skewed
- Contains outliers (infants, elderly)
- Median is robust to outliers


### Why Most Frequent for Embarked?
- Only 2 missing values
- Highly imbalanced categorical feature
- Adding a "missing" category would add noise


In [30]:
numericTransformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categoricalTransformer = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numericTransformer, NUMERIC_FEATURES),
        ("cat", categoricalTransformer, CATEGORICAL_FEATURES)
    ]
)

df[CATEGORICAL_FEATURES]

,Sex,Embarked,Pclass
0,male,S,3
1,female,C,1
2,female,S,3
3,female,S,1
4,male,S,3
...,...,...,...
886,male,S,2
887,female,S,1
888,female,S,3
889,male,C,1


In [31]:
# MODEL 
model = LogisticRegression(
    solver = "liblinear",
    C = 1.0,
    max_iter=100,
    random_state=RANDOM_STATE
)

In [33]:
pipeline = Pipeline(steps = [
    ("preprocessing", preprocessor),
    ("model", model)
])

In [34]:
# Training and MlFlow Logging
with mlflow.start_run():
    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_param("solver", model.solver)
    mlflow.log_param("C", model.C)
    mlflow.log_param("max_iter", model.max_iter)

    pipeline.fit(xTrain, yTrain)
    yPrediction = pipeline.predict(xTest)

    accuracy = accuracy_score(yTest, yPrediction)
    precision = precision_score(yTest, yPrediction)
    recall = recall_score(yTest, yPrediction)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

    # Model Artifact
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model"
    )

2025/12/30 22:51:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [35]:
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

Accuracy  : 0.8045
Precision : 0.7931
Recall    : 0.6667
